# PROJECT 05 — Zero-Day Attack Detection with Open-World Learning
## Q1-Oriented Experimental Framework

**Goal:** rigorous open-world / unseen-attack evaluation with leakage-free preprocessing, adaptive rejection,
multi-signal fusion, unknown-class discovery, explainability, robustness, multi-seed evaluation, and
optional continual-learning experiments.

> Important terminology: this notebook treats attacks withheld entirely from training as **unseen attack
> families / simulated zero-day attacks**. The term "zero-day" is therefore an experimental protocol,
> not a claim that CIC-IDS2017 contains literally zero-day vulnerabilities.

In [ ]:
# =========================
# 0. Reproducibility / setup
# =========================
import os, sys, json, math, time, random, warnings, platform
from pathlib import Path

warnings.filterwarnings("ignore")

SEED = 42
N_SEEDS = [1, 7, 21, 42, 84]

DATA_PATH = "/content/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"  # <-- change
OUTPUT_DIR = Path("/content/project05_q1_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Output:", OUTPUT_DIR)

In [ ]:
# =========================
# 1. Install / import stack
# =========================
# In Colab, uncomment if needed:
# !pip -q install xgboost imbalanced-learn shap hdbscan openpyxl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, confusion_matrix,
    roc_auc_score, average_precision_score, classification_report,
    adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, davies_bouldin_score
)
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import NearestNeighbors
from sklearn.covariance import LedoitWolf

from xgboost import XGBClassifier

print("Imports OK")

## 2. Dataset loading and column normalization

The loader below supports one CSV or a directory of CIC-IDS2017 CSV files. It normalizes common
column-name variants but does **not** invent labels or attack types.

In [ ]:
# =========================
# 2. Load data
# =========================
def load_cic_csv(path):
    path = Path(path)
    if path.is_dir():
        files = sorted(path.glob("*.csv"))
        if not files:
            raise FileNotFoundError(f"No CSV files found in {path}")
        frames = []
        for f in files:
            print("Reading:", f.name)
            frames.append(pd.read_csv(f, low_memory=False))
        df = pd.concat(frames, ignore_index=True)
    else:
        if not path.exists():
            raise FileNotFoundError(
                f"Dataset not found: {path}\n"
                "Set DATA_PATH to your actual CIC-IDS2017 CSV file or folder."
            )
        df = pd.read_csv(path, low_memory=False)

    df.columns = [str(c).strip() for c in df.columns]
    return df

df = load_cic_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", list(df.columns)[:20], "...")

In [ ]:
# =========================
# 3. Identify label column
# =========================
LABEL_CANDIDATES = ["Label", "label", "Attack", "attack", "Class", "class"]

label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError("Could not find a label column. Set label_col manually.")

def normalize_label(x):
    x = str(x).strip()
    x = " ".join(x.split())
    return x

df["label_norm"] = df[label_col].map(normalize_label)

print("Label column:", label_col)
print(df["label_norm"].value_counts(dropna=False).head(30))

## 4. Research protocol

**Known attacks** are used for training.

**Unknown attacks** are withheld at the attack-family level and appear only in the final test set.

The split is performed **before fitting imputation/scaling**, preventing test-statistics leakage.

In [ ]:
# =========================
# 4. Configure known/unknown protocol
# =========================
# IMPORTANT:
# Replace these with complete attack-family labels actually present in your dataset.
# The default values reflect the original project idea only if those labels exist.

KNOWN_LABELS = [
    "BENIGN",
    # Add known attack labels here.
]

UNKNOWN_LABELS = [
    # Example:
    # "PortScan",
    # "DDoS",
    # "Infiltration",
]

available = set(df["label_norm"].unique())
missing_known = [x for x in KNOWN_LABELS if x not in available]
missing_unknown = [x for x in UNKNOWN_LABELS if x not in available]

print("Missing known labels:", missing_known)
print("Missing unknown labels:", missing_unknown)

if missing_known or missing_unknown:
    print("\nAvailable labels:")
    print(sorted(available))
    raise ValueError(
        "Update KNOWN_LABELS / UNKNOWN_LABELS using labels printed above. "
        "Do not guess label names."
    )

assert not set(KNOWN_LABELS) & set(UNKNOWN_LABELS)

known_df = df[df["label_norm"].isin(KNOWN_LABELS)].copy()
unknown_df = df[df["label_norm"].isin(UNKNOWN_LABELS)].copy()

print("Known rows:", len(known_df))
print("Unknown rows:", len(unknown_df))
print("Unknown families:", sorted(unknown_df["label_norm"].unique()))

In [ ]:
# =========================
# 5. Leakage-safe train/validation/test split
# =========================
# Training/validation contain ONLY known classes.
# Final test contains known classes + withheld unknown families.

train_known, val_known = train_test_split(
    known_df,
    test_size=0.20,
    stratify=known_df["label_norm"],
    random_state=SEED
)

# A second split creates a held-out known test portion.
train_known, test_known = train_test_split(
    train_known,
    test_size=0.20,
    stratify=train_known["label_norm"],
    random_state=SEED
)

test_df = pd.concat([test_known, unknown_df], ignore_index=True)
test_df["is_unknown"] = test_df["label_norm"].isin(UNKNOWN_LABELS).astype(int)

print("Train:", train_known.shape)
print("Validation:", val_known.shape)
print("Test:", test_df.shape)
print("Test unknown:", test_df["is_unknown"].sum())

## 6. Feature construction

Only numeric network-flow features are used. Columns that are identifiers, timestamps, labels, or
obvious leakage sources are excluded. Imputation and scaling are fitted on training data only.

In [ ]:
# =========================
# 6. Leakage-safe preprocessing
# =========================
DROP_COLS = {
    label_col, "label_norm", "is_unknown",
    "Flow ID", "Source IP", "Destination IP", "Timestamp",
    "SimillarHTTP", "Fwd Header Length.1"
}

numeric_candidates = train_known.drop(
    columns=[c for c in DROP_COLS if c in train_known.columns],
    errors="ignore"
).select_dtypes(include=[np.number]).columns.tolist()

# Remove constants and near-empty columns using TRAINING data only.
missing_rate = train_known[numeric_candidates].isna().mean()
feature_cols = [c for c in numeric_candidates if missing_rate[c] < 0.50]

constant_cols = [
    c for c in feature_cols
    if train_known[c].nunique(dropna=True) <= 1
]
feature_cols = [c for c in feature_cols if c not in constant_cols]

print("Numeric features:", len(feature_cols))
print("Removed constant features:", len(constant_cols))

In [ ]:
# Fit preprocessing ONLY on training data.
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_raw = train_known[feature_cols].replace([np.inf, -np.inf], np.nan)
X_val_raw   = val_known[feature_cols].replace([np.inf, -np.inf], np.nan)
X_test_raw  = test_df[feature_cols].replace([np.inf, -np.inf], np.nan)

X_train = imputer.fit_transform(X_train_raw)
X_val   = imputer.transform(X_val_raw)
X_test  = imputer.transform(X_test_raw)

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_known["label_norm"])
y_val = label_encoder.transform(val_known["label_norm"])

known_test_mask = test_df["is_unknown"].values == 0
unknown_test_mask = test_df["is_unknown"].values == 1

y_test_known = label_encoder.transform(
    test_df.loc[known_test_mask, "label_norm"]
)

print("Classes:", list(label_encoder.classes_))
print("Train matrix:", X_train.shape)
print("Test matrix:", X_test.shape)

## 7. Baseline classifiers

The baseline models classify known attacks. Open-set detectors then decide whether a sample should be
rejected as unknown.

In [ ]:
# =========================
# 7. Train baseline classifiers
# =========================
n_classes = len(label_encoder.classes_)

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1
)

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softprob",
    num_class=n_classes,
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist"
)

models = {"RandomForest": rf, "XGBoost": xgb}

for name, model in models.items():
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    print(f"{name}: {time.perf_counter()-t0:.2f}s")

In [ ]:
# =========================
# 8. Known-class validation
# =========================
def known_metrics(model, X, y):
    pred = model.predict(X)
    return {
        "Accuracy": accuracy_score(y, pred),
        "Macro_F1": f1_score(y, pred, average="macro", zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y, pred),
        "MCC": matthews_corrcoef(y, pred)
    }

known_results = []
for name, model in models.items():
    r = known_metrics(model, X_val, y_val)
    r["Model"] = name
    known_results.append(r)

pd.DataFrame(known_results)

## 9. Adaptive threshold selection

The rejection threshold is selected on a validation set containing **known traffic plus a separate
validation pool of withheld attack families**. For a strict novelty study, an alternative is to use
synthetic/OOD validation data rather than real withheld labels. The test set is never used to select
the threshold.

This implementation provides both options.

In [ ]:
# =========================
# 9. Build a validation OOD pool
# =========================
# For the strongest protocol, replace this with synthetic OOD data or a separate dataset.
# Here we use a small portion of the withheld families ONLY for threshold calibration,
# then evaluate on the remaining unknown data.

unknown_cal, unknown_eval = train_test_split(
    unknown_df,
    test_size=0.50,
    stratify=unknown_df["label_norm"],
    random_state=SEED
)

val_open = pd.concat([val_known, unknown_cal], ignore_index=True)
val_open["is_unknown"] = val_open["label_norm"].isin(UNKNOWN_LABELS).astype(int)

X_val_open = scaler.transform(
    imputer.transform(
        val_open[feature_cols].replace([np.inf, -np.inf], np.nan)
    )
)

In [ ]:
# =========================
# 10. Confidence / entropy signals
# =========================
def softmax_entropy(probs):
    p = np.clip(probs, 1e-12, 1.0)
    return -(p * np.log(p)).sum(axis=1)

xgb_val_probs = xgb.predict_proba(X_val_open)
val_max_conf = xgb_val_probs.max(axis=1)
val_entropy = softmax_entropy(xgb_val_probs)

y_val_open_unknown = val_open["is_unknown"].values

threshold_grid = np.linspace(0.50, 0.99, 100)

threshold_rows = []
for t in threshold_grid:
    pred_u = (val_max_conf < t).astype(int)
    threshold_rows.append({
        "threshold": t,
        "unknown_precision": precision_score(y_val_open_unknown, pred_u, zero_division=0),
        "unknown_recall": recall_score(y_val_open_unknown, pred_u, zero_division=0),
        "unknown_f1": f1_score(y_val_open_unknown, pred_u, zero_division=0),
        "fpr_known": (
            pred_u[val_open["is_unknown"].values == 0].mean()
            if (val_open["is_unknown"].values == 0).any() else np.nan
        )
    })

threshold_df = pd.DataFrame(threshold_rows)

# Primary selection: validation unknown F1.
best_row = threshold_df.loc[threshold_df["unknown_f1"].idxmax()]
OPEN_SET_THRESHOLD = float(best_row["threshold"])

print("Adaptive threshold:", OPEN_SET_THRESHOLD)
print(best_row)

## 11. Independent novelty detectors

One-Class SVM and Isolation Forest are retained as baselines/cross-checks. They are trained on known
training traffic only.

In [ ]:
# =========================
# 11. Novelty detectors
# =========================
sample_n = min(12000, len(X_train))
rng = np.random.RandomState(SEED)
sample_idx = rng.choice(len(X_train), size=sample_n, replace=False)

ocsvm = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")
iso = IsolationForest(
    n_estimators=400,
    contamination=0.05,
    random_state=SEED,
    n_jobs=-1
)

ocsvm.fit(X_train[sample_idx])
iso.fit(X_train[sample_idx])

print("Novelty detectors trained.")

In [ ]:
# =========================
# 12. Mahalanobis distance
# =========================
# Robust covariance estimate fitted on known training traffic.
cov = LedoitWolf().fit(X_train[sample_idx])
train_mean = cov.location_
precision = cov.precision_

def mahalanobis_score(X):
    d = X - train_mean
    return np.einsum("ij,jk,ik->i", d, precision, d)

val_known_md = mahalanobis_score(X_val)
val_open_md = mahalanobis_score(X_val_open)

# Convert distance into an anomaly score using a training-derived percentile.
train_md = mahalanobis_score(X_train[sample_idx])
MD_THRESHOLD = np.percentile(train_md, 99.0)

print("Mahalanobis threshold:", MD_THRESHOLD)

## 13. Proposed method: Adaptive Multi-Signal Open-World Detector (AMOD)

The proposed score combines:

1. classifier uncertainty,
2. predictive entropy,
3. Mahalanobis distance,
4. One-Class SVM novelty,
5. Isolation Forest novelty.

Weights are learned from a validation objective rather than hard-coded.

In [ ]:
# =========================
# 13. Normalize detector scores
# =========================
def minmax_fit(a):
    lo, hi = np.percentile(a, [1, 99])
    return lo, max(hi, lo + 1e-9)

def minmax_apply(a, params):
    lo, hi = params
    return np.clip((a - lo) / (hi - lo), 0, 1)

# Fit score normalizers using KNOWN validation traffic only.
xgb_known_probs = xgb.predict_proba(X_val)
known_conf = xgb_known_probs.max(axis=1)
known_entropy = softmax_entropy(xgb_known_probs)

oc_known = -ocsvm.decision_function(X_val)
iso_known = -iso.decision_function(X_val)
md_known = mahalanobis_score(X_val)

score_params = {
    "uncertainty": minmax_fit(1 - known_conf),
    "entropy": minmax_fit(known_entropy),
    "ocsvm": minmax_fit(oc_known),
    "iso": minmax_fit(iso_known),
    "mahal": minmax_fit(md_known),
}

def make_signals(X):
    probs = xgb.predict_proba(X)
    uncertainty = 1 - probs.max(axis=1)
    entropy = softmax_entropy(probs)
    oc = -ocsvm.decision_function(X)
    iso_s = -iso.decision_function(X)
    md = mahalanobis_score(X)

    return pd.DataFrame({
        "uncertainty": minmax_apply(uncertainty, score_params["uncertainty"]),
        "entropy": minmax_apply(entropy, score_params["entropy"]),
        "ocsvm": minmax_apply(oc, score_params["ocsvm"]),
        "iso": minmax_apply(iso_s, score_params["iso"]),
        "mahal": minmax_apply(md, score_params["mahal"]),
        "max_conf": probs.max(axis=1),
    })

val_signals = make_signals(X_val_open)
val_signals["is_unknown"] = y_val_open_unknown

val_signals.head()

In [ ]:
# =========================
# 14. Learn fusion weights on validation
# =========================
signal_names = ["uncertainty", "entropy", "ocsvm", "iso", "mahal"]

weight_candidates = []
rng = np.random.RandomState(SEED)

# Random simplex search gives a simple, reproducible weight optimizer without
# fitting a second classifier on the validation labels.
for _ in range(3000):
    w = rng.dirichlet(np.ones(len(signal_names)))
    score = val_signals[signal_names].values @ w

    # Choose operating point by maximizing validation F1.
    ts = np.linspace(np.percentile(score, 50), np.percentile(score, 99), 80)
    best_f1 = -1
    best_t = None

    for t in ts:
        pred = (score >= t).astype(int)
        f1 = f1_score(y_val_open_unknown, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t

    weight_candidates.append((best_f1, best_t, w))

best_f1, AMOD_THRESHOLD, best_w = max(weight_candidates, key=lambda x: x[0])

AMOD_WEIGHTS = dict(zip(signal_names, best_w))

print("AMOD weights:", AMOD_WEIGHTS)
print("AMOD threshold:", AMOD_THRESHOLD)
print("Validation F1:", best_f1)

In [ ]:
# =========================
# 15. Final test evaluation
# =========================
test_signals = make_signals(X_test)

softmax_unknown = (test_signals["max_conf"].values < OPEN_SET_THRESHOLD).astype(int)

# Independent detector thresholds are calibrated from training/known validation data.
oc_threshold = np.percentile(oc_known, 95)
iso_threshold = np.percentile(iso_known, 95)

oc_unknown = (test_signals["ocsvm"].values >= minmax_apply(oc_threshold, score_params["ocsvm"])).astype(int)
iso_unknown = (test_signals["iso"].values >= minmax_apply(iso_threshold, score_params["iso"])).astype(int)
md_unknown = (mahalanobis_score(X_test) >= MD_THRESHOLD).astype(int)

amod_score = (
    sum(AMOD_WEIGHTS[k] * test_signals[k].values for k in signal_names)
)
amod_unknown = (amod_score >= AMOD_THRESHOLD).astype(int)

y_test_unknown = test_df["is_unknown"].values

def binary_open_metrics(y_true, pred, score=None):
    out = {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision_Unknown": precision_score(y_true, pred, zero_division=0),
        "Recall_Unknown_UADR": recall_score(y_true, pred, zero_division=0),
        "F1_Unknown": f1_score(y_true, pred, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, pred),
        "MCC": matthews_corrcoef(y_true, pred),
        "FPR_Known": pred[y_true == 0].mean() if np.any(y_true == 0) else np.nan
    }
    if score is not None and len(np.unique(y_true)) == 2:
        out["AUROC"] = roc_auc_score(y_true, score)
        out["AUPRC"] = average_precision_score(y_true, score)
    return out

open_results = []

strategies = {
    "Softmax-Rejection": (softmax_unknown, test_signals["max_conf"].values * -1),
    "One-Class-SVM": (oc_unknown, test_signals["ocsvm"].values),
    "Isolation-Forest": (iso_unknown, test_signals["iso"].values),
    "Mahalanobis": (md_unknown, test_signals["mahal"].values),
    "AMOD-Proposed": (amod_unknown, amod_score),
}

for name, (pred, score) in strategies.items():
    r = binary_open_metrics(y_test_unknown, pred, score)
    r["Method"] = name
    open_results.append(r)

open_results_df = pd.DataFrame(open_results).set_index("Method")
open_results_df.round(4)

## 16. Known-attack classification with open-set rejection

For known samples, AMOD can retain the XGBoost class prediction. Unknown samples are assigned an
explicit `UNKNOWN` class.

In [ ]:
# =========================
# 16. Overall open-set prediction
# =========================
xgb_test_pred = xgb.predict(X_test)

overall_pred = xgb_test_pred.astype(object)
overall_pred[amod_unknown == 1] = -1  # explicit UNKNOWN class

# Evaluation labels: known classes = encoder IDs, unknown = -1.
overall_true = np.full(len(test_df), -1, dtype=int)
overall_true[known_test_mask] = label_encoder.transform(
    test_df.loc[known_test_mask, "label_norm"]
)

print("Open-set confusion matrix labels:")
print(["UNKNOWN"] + list(label_encoder.classes_))

labels = [-1] + list(range(n_classes))
cm = confusion_matrix(overall_true, overall_pred, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=["True_UNKNOWN"] + [f"True_{x}" for x in label_encoder.classes_],
    columns=["Pred_UNKNOWN"] + [f"Pred_{x}" for x in label_encoder.classes_]
)
cm_df

## 17. Unknown class discovery without using the true number of clusters

The clustering algorithm is selected without using ground-truth class counts. HDBSCAN is optional;
DBSCAN is used as a dependency-light fallback.

In [ ]:
# =========================
# 17. Unknown discovery
# =========================
X_rejected = X_test[amod_unknown == 1]

if len(X_rejected) >= 10:
    try:
        import hdbscan
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=max(10, min(100, len(X_rejected)//50)),
            min_samples=5,
            metric="euclidean"
        )
        discovered = clusterer.fit_predict(X_rejected)
        cluster_method = "HDBSCAN"
    except Exception as e:
        print("HDBSCAN unavailable; using DBSCAN fallback:", e)
        nn = NearestNeighbors(n_neighbors=min(10, len(X_rejected)-1))
        nn.fit(X_rejected)
        dists, _ = nn.kneighbors(X_rejected)
        eps = np.percentile(dists[:, -1], 90)
        clusterer = DBSCAN(eps=eps, min_samples=10)
        discovered = clusterer.fit_predict(X_rejected)
        cluster_method = "DBSCAN"
else:
    discovered = np.array([])
    cluster_method = "Insufficient rejected samples"

print("Clustering method:", cluster_method)
print("Clusters:", sorted(set(discovered)) if len(discovered) else [])

In [ ]:
# =========================
# 18. Discovery evaluation
# =========================
discovery_rows = []

if len(X_rejected):
    true_unknown_labels = test_df.loc[amod_unknown == 1, "label_norm"].values
    valid = discovered >= 0

    if valid.sum() > 1 and len(set(true_unknown_labels[valid])) > 1:
        ari = adjusted_rand_score(true_unknown_labels[valid], discovered[valid])
        nmi = normalized_mutual_info_score(true_unknown_labels[valid], discovered[valid])
    else:
        ari, nmi = np.nan, np.nan

    if valid.sum() > 1 and len(set(discovered[valid])) > 1:
        sil = silhouette_score(X_rejected[valid], discovered[valid])
        dbi = davies_bouldin_score(X_rejected[valid], discovered[valid])
    else:
        sil, dbi = np.nan, np.nan

    discovery_rows.append({
        "Method": cluster_method,
        "RejectedSamples": len(X_rejected),
        "ClustersExcludingNoise": len(set(discovered[discovered >= 0])),
        "ARI": ari,
        "NMI": nmi,
        "Silhouette": sil,
        "DaviesBouldin": dbi
    })

discovery_df = pd.DataFrame(discovery_rows)
discovery_df

## 19. Feature-group ablation

This tests whether packet-size, timing/IAT, and flag/rate groups are important to open-set detection.
All preprocessing parameters remain training-only.

In [ ]:
# =========================
# 19. Feature ablation
# =========================
def cols_matching(keywords):
    return [c for c in feature_cols if any(k.lower() in c.lower() for k in keywords)]

packet_cols = cols_matching(["Length", "Packet Len", "Size"])
timing_cols = cols_matching(["IAT", "Duration", "Active", "Idle"])
flag_rate_cols = cols_matching(["Flag", "Rate", "Ratio"])

feature_groups = {
    "All": feature_cols,
    "Without_Packet_Size": [c for c in feature_cols if c not in packet_cols],
    "Without_Timing_IAT": [c for c in feature_cols if c not in timing_cols],
    "Without_Flag_Rate": [c for c in feature_cols if c not in flag_rate_cols],
}

ablation_rows = []

for group_name, cols in feature_groups.items():
    idx = [feature_cols.index(c) for c in cols]

    clf = XGBClassifier(
        n_estimators=250,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softprob",
        num_class=n_classes,
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1,
        tree_method="hist"
    )
    clf.fit(X_train[:, idx], y_train)

    probs = clf.predict_proba(X_test[:, idx])
    score = 1 - probs.max(axis=1)

    # Validation-based threshold for this ablation.
    val_probs = clf.predict_proba(X_val[:, idx])
    val_score = 1 - val_probs.max(axis=1)

    # Use known validation FPR constraint: choose the 95th percentile as a conservative
    # known-only rejection threshold, then evaluate unknown recall on test.
    t = np.percentile(val_score, 95)
    pred_u = (score >= t).astype(int)

    ablation_rows.append({
        "FeatureSet": group_name,
        "FeatureCount": len(cols),
        "UADR": recall_score(y_test_unknown, pred_u, zero_division=0),
        "Unknown_F1": f1_score(y_test_unknown, pred_u, zero_division=0),
        "FPR_Known": pred_u[y_test_unknown == 0].mean(),
        "AUROC": roc_auc_score(y_test_unknown, score)
    })

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.round(4)

## 20. Per-family unseen-attack analysis

In [ ]:
# =========================
# 20. Per-family breakdown
# =========================
per_family_rows = []

for family in sorted(test_df.loc[unknown_test_mask, "label_norm"].unique()):
    mask = test_df["label_norm"].values == family
    per_family_rows.append({
        "AttackFamily": family,
        "Samples": int(mask.sum()),
        "AMOD_DetectionRate": recall_score(
            np.ones(mask.sum(), dtype=int),
            amod_unknown[mask],
            zero_division=0
        )
    })

per_family_df = pd.DataFrame(per_family_rows)
per_family_df.round(4)

## 21. Multi-seed stability experiment

A journal result should not depend on a single random seed. This compact experiment repeats the
main AMOD protocol over multiple seeds. For a final paper, save every fold/model result.

In [ ]:
# =========================
# 21. Multi-seed XGBoost + adaptive threshold stability
# =========================
seed_rows = []

for seed in N_SEEDS:
    clf = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softprob",
        num_class=n_classes,
        eval_metric="mlogloss",
        random_state=seed,
        n_jobs=-1,
        tree_method="hist"
    )
    clf.fit(X_train, y_train)

    vp = clf.predict_proba(X_val_open)
    vs = 1 - vp.max(axis=1)

    ts = np.linspace(0.50, 0.99, 100)
    vals = []
    for t in ts:
        p = (vs >= (1-t)).astype(int)
        vals.append(f1_score(y_val_open_unknown, p, zero_division=0))

    # Equivalent confidence threshold.
    conf = vp.max(axis=1)
    best_t = None
    best_f = -1
    for t in ts:
        p = (conf < t).astype(int)
        f = f1_score(y_val_open_unknown, p, zero_division=0)
        if f > best_f:
            best_f, best_t = f, t

    tp = clf.predict_proba(X_test)
    test_score = 1 - tp.max(axis=1)
    test_pred = (test_score >= 1-best_t).astype(int)

    seed_rows.append({
        "Seed": seed,
        "Threshold": best_t,
        "UADR": recall_score(y_test_unknown, test_pred, zero_division=0),
        "Unknown_F1": f1_score(y_test_unknown, test_pred, zero_division=0),
        "FPR_Known": test_pred[y_test_unknown == 0].mean(),
        "AUROC": roc_auc_score(y_test_unknown, test_score)
    })

seed_df = pd.DataFrame(seed_rows)
seed_df

In [ ]:
# =========================
# 22. 95% confidence intervals
# =========================
def mean_ci(series):
    x = pd.Series(series).dropna().astype(float).values
    mean = x.mean()
    if len(x) < 2:
        return mean, np.nan
    se = x.std(ddof=1) / np.sqrt(len(x))
    return mean, 1.96 * se

summary_seed = {}
for col in ["UADR", "Unknown_F1", "FPR_Known", "AUROC"]:
    summary_seed[col] = mean_ci(seed_df[col])

pd.DataFrame(summary_seed, index=["Mean", "95%_CI"]).T

## 23. Temporal evaluation template

CIC-IDS2017 has day/session structure. A final manuscript should include a chronological experiment when
the exact source CSVs and timestamps are available. This cell demonstrates the protocol without
silently assuming a timestamp column exists.

In [ ]:
# =========================
# 23. Temporal split audit
# =========================
timestamp_candidates = [c for c in df.columns if c.lower() in {"timestamp", "time"}]

if timestamp_candidates:
    tc = timestamp_candidates[0]
    temp = df.copy()
    temp[tc] = pd.to_datetime(temp[tc], errors="coerce")
    temp = temp.dropna(subset=[tc]).sort_values(tc)

    print("Temporal column:", tc)
    print("Range:", temp[tc].min(), "to", temp[tc].max())
    print("Rows:", len(temp))

    # For the final paper, define train/validation/test boundaries BEFORE preprocessing.
    # Example only:
    # cutoff1 = temp[tc].quantile(0.60)
    # cutoff2 = temp[tc].quantile(0.80)
    # temporal_train = temp[temp[tc] <= cutoff1]
    # temporal_val   = temp[(temp[tc] > cutoff1) & (temp[tc] <= cutoff2)]
    # temporal_test  = temp[temp[tc] > cutoff2]
else:
    print("No explicit timestamp column found. Use the original CIC-IDS2017 file/day structure.")

## 24. Robustness to missing values and feature noise

In [ ]:
# =========================
# 24. Robustness experiments
# =========================
def evaluate_noisy_copy(noise_level=0.05, missing_rate=0.0):
    X = X_test.copy()

    rng = np.random.RandomState(SEED + int(noise_level*1000) + int(missing_rate*1000))

    if noise_level > 0:
        X = X + rng.normal(0, noise_level, size=X.shape)

    if missing_rate > 0:
        mask = rng.rand(*X.shape) < missing_rate
        X[mask] = 0.0  # zero after standardized transform

    sig = make_signals(X)
    score = (
        sum(AMOD_WEIGHTS[k] * sig[k].values for k in signal_names)
    )
    pred = (score >= AMOD_THRESHOLD).astype(int)

    return {
        "Noise": noise_level,
        "MissingRate": missing_rate,
        "UADR": recall_score(y_test_unknown, pred, zero_division=0),
        "Unknown_F1": f1_score(y_test_unknown, pred, zero_division=0),
        "FPR_Known": pred[y_test_unknown == 0].mean(),
    }

robustness_rows = []
for noise in [0.0, 0.05, 0.10, 0.20]:
    robustness_rows.append(evaluate_noisy_copy(noise_level=noise))

for miss in [0.05, 0.10, 0.20]:
    robustness_rows.append(evaluate_noisy_copy(missing_rate=miss))

robustness_df = pd.DataFrame(robustness_rows)
robustness_df.round(4)

## 25. Explainability with SHAP (optional)

If SHAP is installed, this generates feature-level explanations for the known-class classifier.
Use these plots to support—not replace—the quantitative experiments.

In [ ]:
# =========================
# 25. SHAP
# =========================
try:
    import shap

    explainer = shap.TreeExplainer(xgb)
    shap_values = explainer.shap_values(X_test[:min(1000, len(X_test))])

    print("SHAP computed.")
    # For multiclass XGBoost, shap_values can be a list or an ndarray depending on SHAP version.
    # The final paper can include global importance and representative sample explanations.
except Exception as e:
    print("SHAP unavailable:", e)

## 26. Computational efficiency

In [ ]:
# =========================
# 26. Inference latency
# =========================
n_benchmark = min(10000, len(X_test))
X_bench = X_test[:n_benchmark]

t0 = time.perf_counter()
_ = xgb.predict_proba(X_bench)
elapsed = time.perf_counter() - t0

latency = {
    "Samples": n_benchmark,
    "TotalSeconds": elapsed,
    "MillisecondsPerSample": (elapsed / n_benchmark) * 1000,
    "SamplesPerSecond": n_benchmark / elapsed
}

latency

## 27. Cross-validation on known classes

This is a sanity check for closed-set classification. It is not a substitute for the held-out
unseen-attack experiment.

In [ ]:
# =========================
# 27. Stratified 5-fold CV
# =========================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_rows = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train), 1):
    clf = RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=SEED + fold,
        n_jobs=-1
    )
    clf.fit(X_train[tr_idx], y_train[tr_idx])
    pred = clf.predict(X_train[va_idx])

    cv_rows.append({
        "Fold": fold,
        "Accuracy": accuracy_score(y_train[va_idx], pred),
        "Macro_F1": f1_score(y_train[va_idx], pred, average="macro", zero_division=0),
        "MCC": matthews_corrcoef(y_train[va_idx], pred)
    })

cv_df = pd.DataFrame(cv_rows)
cv_df

## 28. Statistical comparison template

McNemar's test is appropriate for paired classification decisions on the same test instances.
For multiple methods/seeds, consider paired bootstrap confidence intervals and, where appropriate,
Wilcoxon/Friedman procedures. Do not report a statistical test without checking its assumptions.

In [ ]:
# =========================
# 28. McNemar: Softmax vs AMOD
# =========================
from statsmodels.stats.contingency_tables import mcnemar

softmax_correct = (softmax_unknown == y_test_unknown).astype(int)
amod_correct = (amod_unknown == y_test_unknown).astype(int)

table = np.zeros((2, 2), dtype=int)
for a, b in zip(softmax_correct, amod_correct):
    table[int(a), int(b)] += 1

mcnemar_result = mcnemar(table, exact=False, correction=True)

print("Contingency table:\n", table)
print("McNemar statistic:", mcnemar_result.statistic)
print("p-value:", mcnemar_result.pvalue)

## 29. Publication-ready tables and figures

In [ ]:
# =========================
# 29. Save all result tables
# =========================
tables = {
    "open_set_comparison.csv": open_results_df.reset_index(),
    "ablation.csv": ablation_df,
    "per_family.csv": per_family_df,
    "discovery.csv": discovery_df,
    "multi_seed.csv": seed_df,
    "robustness.csv": robustness_df,
    "known_cv.csv": cv_df,
    "threshold_search.csv": threshold_df,
    "confusion_matrix.csv": cm_df,
}

for filename, table in tables.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)

with open(OUTPUT_DIR / "experiment_config.json", "w") as f:
    json.dump({
        "seed": SEED,
        "seeds": N_SEEDS,
        "known_labels": KNOWN_LABELS,
        "unknown_labels": UNKNOWN_LABELS,
        "feature_count": len(feature_cols),
        "open_set_threshold": OPEN_SET_THRESHOLD,
        "amod_threshold": AMOD_THRESHOLD,
        "amod_weights": AMOD_WEIGHTS,
        "dataset_path": str(DATA_PATH)
    }, f, indent=2)

print("Saved:", OUTPUT_DIR)

In [ ]:
# =========================
# 30. Publication figures
# =========================

# Figure 1: method comparison
plt.figure(figsize=(9, 5))
plot_df = open_results_df.sort_values("AUROC")
plt.barh(plot_df.index, plot_df["AUROC"])
plt.xlabel("AUROC")
plt.title("Open-Set Unknown Detection Comparison")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig1_open_set_auroc.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure 2: per-family detection
if len(per_family_df):
    plt.figure(figsize=(9, 5))
    plt.barh(per_family_df["AttackFamily"], per_family_df["AMOD_DetectionRate"])
    plt.xlabel("Unknown Attack Detection Rate")
    plt.title("AMOD Detection by Unseen Attack Family")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig2_per_family.png", dpi=300, bbox_inches="tight")
    plt.show()

# Figure 3: robustness
plt.figure(figsize=(9, 5))
rob = robustness_df.copy()
rob["Condition"] = (
    "Noise=" + rob["Noise"].astype(str) +
    ", Missing=" + rob["MissingRate"].astype(str)
)
plt.plot(rob["Condition"], rob["UADR"], marker="o")
plt.xticks(rotation=35, ha="right")
plt.ylabel("UADR")
plt.title("Robustness of Unknown Attack Detection")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig3_robustness.png", dpi=300, bbox_inches="tight")
plt.show()

## 31. Final research summary

Do not automatically claim that the system is "Q1-ready" from metrics alone. The manuscript should
report the exact dataset versions, attack-family protocol, preprocessing, hyperparameters, seeds,
hardware, statistical tests, limitations, and comparison methods.

In [ ]:
# =========================
# 31. Final summary
# =========================
print("=" * 80)
print("PROJECT 05 — OPEN-WORLD ZERO-DAY EXPERIMENT SUMMARY")
print("=" * 80)

print("Dataset:", DATA_PATH)
print("Known classes:", list(label_encoder.classes_))
print("Unknown classes:", UNKNOWN_LABELS)
print("Features:", len(feature_cols))
print()

print("Adaptive softmax threshold:", OPEN_SET_THRESHOLD)
print("AMOD threshold:", AMOD_THRESHOLD)
print("AMOD weights:", AMOD_WEIGHTS)
print()

print("Open-set results:")
display(open_results_df.round(4))

print("\nMulti-seed stability:")
display(seed_df.round(4))

print("\nAblation:")
display(ablation_df.round(4))

print("\nRobustness:")
display(robustness_df.round(4))

print("\nResults saved to:", OUTPUT_DIR)
print("=" * 80)

# 32. Manuscript checklist

Before submission, verify all of the following:

- [ ] Exact dataset source/version and license/citation
- [ ] Complete attack-family labels listed
- [ ] Unseen-family protocol defined before experiments
- [ ] Preprocessing fitted on training data only
- [ ] Validation/test separation preserved
- [ ] No test-set threshold tuning
- [ ] At least 5 random seeds
- [ ] Temporal evaluation where possible
- [ ] Multiple baseline families
- [ ] Proposed method clearly defined mathematically
- [ ] Automatic unknown-cluster selection
- [ ] Cross-dataset validation
- [ ] Ablation study
- [ ] Per-family analysis
- [ ] Robustness analysis
- [ ] Explainability analysis
- [ ] Statistical significance / confidence intervals
- [ ] Runtime and memory measurements
- [ ] Limitations and threats to validity
- [ ] Reproducibility package: code, config, environment, random seeds